# Gray–Scott · (f, k) phase diagram

Exploratory analysis of The Well gray_scott_reaction_diffusion feature table.
**Not ANOVA** — focus on the feed–kill phase map and pattern metrics vs parameters.


In [ ]:
from pathlib import Path
import sys
ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.dataset_catalog import get_dataset

spec = get_dataset('gray_scott')
df = pd.read_parquet(spec.feature_path)
print(df.shape)
df.head()


## Sparse (f, k) factorial / phase diagram

Six named regimes occupy discrete cells on the feed–kill plane.

In [ ]:
cell = (
    df.groupby(['f', 'k', 'pattern'], as_index=False)['pattern_contrast']
    .mean()
    .sort_values(['k', 'f'])
)
display(cell)

fig, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(cell['f'], cell['k'], c=cell['pattern_contrast'], s=200 + 400*cell['pattern_contrast'],
                cmap='YlGn', edgecolors='k')
for _, r in cell.iterrows():
    ax.annotate(r['pattern'], (r['f'], r['k']), textcoords='offset points', xytext=(6, 6), fontsize=9)
ax.set_xlabel('feed f'); ax.set_ylabel('kill k')
ax.set_title('Gray–Scott phase diagram — mean pattern_contrast')
fig.colorbar(sc, ax=ax, label='pattern_contrast')
plt.tight_layout(); plt.show()


## Pattern metrics vs parameters

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.boxplot(data=df, x='pattern', y='pattern_contrast', ax=axes[0], color='#8A9470')
axes[0].tick_params(axis='x', rotation=30)
axes[0].set_title('Contrast by regime')
sns.scatterplot(data=df, x='f', y='mean_A', hue='pattern', ax=axes[1])
axes[1].set_title('mean_A vs feed f')
plt.tight_layout(); plt.show()


In [ ]:
num = [c for c in ['f','k','pattern_contrast','mean_A','mean_B','std_A','std_B','spectral_slope','time_to_steady'] if c in df.columns]
corr = df[num].corr(method='spearman')
plt.figure(figsize=(7, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0)
plt.title('Spearman correlations — params & metrics')
plt.tight_layout(); plt.show()


## Takeaway

Treat (f, k) as a **phase diagram / sparse factorial**, not a classical balanced ANOVA layout. Regime separation in contrast and concentration is the primary signal.